#### Prepare dataset for fine-tune a pretrained model. 

In [1]:
import pandas as pd
import numpy as np
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import DataFrameLoader

In [2]:
data = pd.read_csv("cleaned_data.csv",index_col=0)

In [3]:
category = data["sub_category"].unique()

In [ ]:
category

In [5]:
df = pd.DataFrame()
df = pd.concat([df,data[data["sub_category"]==category[0]]])
df = pd.concat([df,data[data["sub_category"]==category[1]].sample(frac=0.50,random_state=50)])
df = pd.concat([df,data[data["sub_category"]==category[2]].sample(frac=0.50,random_state=50)])

In [ ]:
df

In [7]:
df.drop("main_category",axis=1,inplace=True)

In [ ]:
df.groupby("sub_category").count()

In [9]:
df["name"]=df["name"].apply(lambda x:str(x).replace(',...','') if str(x).endswith(',...') else (str(x).replace(", ...","") if str(x).endswith(", ...") else str(x)))
f_1 = pd.DataFrame(df["name"].apply(lambda x: str(x) if str(x).endswith("...") else None)).dropna()
f_0 = pd.DataFrame(f_1["name"].apply(lambda x: str(x) if str(x).split()[-1].strip()[:-3].isalpha() or str(x).endswith(" ...") or not str(x).split()[-1].strip()[-7:-4].isupper() else None)).dropna()
df.loc[(f_1.drop(f_0.index)).index, "name"] = df["name"].apply(lambda x:(' '.join(str(x).split()[:-1])))
df["name"] = df["name"].apply(lambda x:str(x).replace("...","") if str(x).endswith("...") else str(x))

In [ ]:
f_1

In [11]:
for i in f_0.index:
    f_0.loc[i,"sub_category"] = df["sub_category"][i]

In [ ]:
f_0.shape

In [ ]:
df.shape

In [14]:
df.loc[df.index,"product_name"] = np.where((df["sub_category"]==category[0]), df['name'].apply(lambda x: str(x).split("(")[0] if str(x).find("(") else str(x)),df["name"])
df.loc[df.index,"product_name"] = np.where((df["sub_category"]==category[0]),df["product_name"].apply(lambda x: str(x).split(",")[0] if str(x).count(",")>=2 else str(x)),df["product_name"])

In [15]:
def func(data, name, product_name):
    for i in data.index:
        if data.at[i, "sub_category"] == category[0]:  # Check the condition for col0
            # Perform the string split operation and assign the modified value
            data.at[i, "description"] = data.at[i, name].replace(data.at[i, product_name], "")
        else:
            data.at[i, "description"] = None
    return data

In [16]:
df = func(df,"name","product_name")

In [ ]:
df[df["sub_category"]==category[0]].shape

In [ ]:
loader = DataFrameLoader(df, page_content_column="name")
docs=loader.load()
len(docs)

In [ ]:
embedding_model = OllamaEmbeddings(model='nomic-embed-text')
embed_1 = FAISS.from_documents(docs,embedding_model)

In [23]:
from langchain.prompts import PromptTemplate
from langchain_community.llms import Ollama
retriever = embed_1.as_retriever(search_type ='similarity',search_kwargs={'k': 20})
llm=Ollama(model='llama3')

In [24]:
sentences = {}
prompt = PromptTemplate(
    input_variables=["incomplete_sentence","retrieved_context"],
    template="Complete the following sentence: {incomplete_sentence} do not generate extra sentence, word and return full complete sentence, Use the context: {retrieved_context}."
)

# Initialize the LLM 
llm = llm=Ollama(model='llama3')

#completion_chain = LLMChain(llm=llm, prompt=prompt)
completion_chain= prompt | llm

# Step 5: Define the input (incomplete sentence)
for idx, sentence in f_0["name"].to_dict().items():
    incomplete_sentence = sentence

    # Step 6: Retrieve relevant context from the retriever
    retrieved_docs = retriever.invoke(incomplete_sentence)

    retrieved_context = "\n".join([doc.metadata["description"] if doc.metadata["description"]!= None else doc.page_content for doc in retrieved_docs])



    # Step 7: Combine the input with the retrieved context
    result = completion_chain.invoke({"incomplete_sentence": incomplete_sentence, "retrieved_context": retrieved_context})
    sentences.update({idx:result})

In [71]:
filter_senteces = {}
for key, value in sentences.items():
    filter_value = (value.split("\n\n").pop())
    filter_senteces.update({key:filter_value})
filter_df = pd.DataFrame.from_dict(filter_senteces,orient="index",columns=["name"])

In [74]:
from difflib import SequenceMatcher
def combine_sentences_dynamic(sentence1, sentence2):
    # Step 1: Clean the sentences by removing ellipses and trimming spaces
    
    sentence1 = sentence1.replace("...", "")
    sentence2 = sentence2.replace("...", "")
    
    # Step 2: Tokenize sentences into words
    words1 = sentence1.split()
    words2 = sentence2.split()

    # Step 3: Find the largest overlap using SequenceMatcher
    matcher = SequenceMatcher(None, words1, words2)
    match = matcher.find_longest_match(0, len(words1), 0, len(words2))

    # Step 4: Combine sentences based on overlap
    if match.size > 0:  # If there is an overlap
        overlap_start = match.a
        overlap_end = match.a + match.size
        
        # Combine sentences with overlap
        combined = " ".join(words1[:overlap_start] + words2)
    else:
        # If no overlap, simply concatenate
        combined = f"{sentence1} {sentence2}"
    
    return combined

In [111]:
combined_sentence = {}
for i in filter_df.index:
    combined = combine_sentences_dynamic(f_0["name"][i],filter_df["name"][i])
    combined_sentence.update({i:combined})
combined_df = pd.DataFrame.from_dict(combined_sentence,orient="index",columns=["name"])


In [133]:
dataset = df.copy()
dataset.loc[combined_df.index, "name"] = combined_df["name"]
dataset.drop(["product_name","description"], axis=1, inplace=True)

In [140]:
dataset.loc[dataset.index,"product_name"] = np.where((dataset["sub_category"]==category[0]), dataset['name'].apply(lambda x: str(x).split("(")[0] if str(x).find("(") else str(x)),dataset["name"])
dataset.loc[dataset.index,"product_name"] = np.where((dataset["sub_category"]==category[0]),dataset["product_name"].apply(lambda x: str(x).split(",")[0] if str(x).count(",")>=2 else str(x)),dataset["product_name"])

In [141]:
dataset = func(dataset,"name","product_name")

In [142]:
dataset["actual_price_new"] = dataset["actual_price"].apply(lambda x:str(x).replace("₹","")).astype(float)
dataset["discount_price_new"] = dataset["discount_price"].apply(lambda x:str(x).replace("₹","")).astype(float)

#### Create some category-specific query-response pairs to fine-tune a pretrained LLM  

#### 1. Tell me about product 

In [143]:
query1=[]
response1 = []
category1 = []
def query_responsepair1(product_name,description,name,row,category):
    query = f"Can you tell me about {row[product_name]} this product?"
    category=f"{row[category]}"
    if row[description ] != None:
        response = f"""Absolutely! This product has some impressive features-lets see the features.\n\nThe {row[product_name]} come with {row[description]}.
\nwould you like to know more information, such as price, ratings etc or about any other product?"""
        query1.append(query)
        response1.append(response)
        category1.append(category)
    elif row[description ] == None:
        response = f"""Absolutely! This product has some impressive features-lets see the product.\n\n {row[name]}.
\nwould you like to know more information, such as price, ratings etc or about any other product?"""
        query1.append(query)
        response1.append(response)
        category1.append(category)
dataset.apply(lambda x: query_responsepair1("product_name","description","name",x,"sub_category"), axis=1)
query_response_pair_1 = pd.DataFrame({"Query": query1, "Response": response1, "category": category1})

##### What is actual price of product?

In [145]:
query2 = []
response2 = []
dis_range = []
category2 = []
def query_responsepair2(product_name,actual,percent,discount,row,category):
    query = f"what is actual price of {row[product_name]}?"
    act_price = row[actual]
    dis_price = row[discount]
    dis = row[percent]
    category=f"{row[category]}"
    if dis_price and dis > 0 and dis < 40:
        response = f"The price of the {row[product_name]} is {act_price}. This product is now available in {dis_price} with {dis}% discount.\nWould you like to add it to your cart or looking for other option?"
        response2.append(response)
        dis_range.append("0<dis<40")
        category2.append(category)
    elif dis_price and dis >=40:
        response = f"The price of the {row[product_name]} is {act_price}. This product is now available for {dis_price} with an incredible {dis}% off as part of our festive sale!\nDon't miss out on this limited-time offer! Would you like to add it to your cart or are you interested in exploring other options?"
        response2.append(response)
        dis_range.append("dis>=40")
        category2.append(category)
    else:
        response = f"The price of the {row[product_name]} is {act_price}. There is no discount available. Would you like to add it to your cart or are you interested in exploring other options?"
        response2.append(response)
        dis_range.append("dis==0")
        category2.append(category)

    query2.append(query)


dataset.apply(lambda x:query_responsepair2(discount="discount_price",actual="actual_price",percent="discount_%",product_name="product_name",row=x,category="sub_category"), axis=1)    
query_response_pair_2 = pd.DataFrame({"Query": query2, "Response": response2, "category":category2, "discount_range":dis_range})   

##### 3. Can i get extra discount on this product?

In [147]:
query3 = []
response3 = []
category3 = []
def query_responsepair3(product_name,percent,row,category):
    query = f"Can i get extra discount on this product {row[product_name]}?"
    dis = row[percent]
    category=f"{row[category]}"
    if dis == 0:
        response = f"""Yes, you can get 5% off on this product using coupon code.
                    """
        response3.append(response)
        category3.append(category)
    elif dis > 0 and dis <= 20:
        response = f"""Yes, you can get extra 5% off on this product if you are new user.\nBut if you are already an user then you can get upto 10% off.
                    """
        response3.append(response)
        category3.append(category)
    elif dis > 20 and dis <= 39:
        response = f"Yes, you can get extra 2% off on this product if you are new user.\nBut if you are already an user then you can get upto 5% off."
        response3.append(response)
        category3.append(category)
    elif dis >= 40:
        response = f"""Sorry! This product has already been discounted by or over 30%, so there is no further discount possible as per our product guidelines.\nIf you Looking for bigger discounts, Check out our ongoing festival sale with offers up to 60% off!"""
        response3.append(response)
        category3.append(category)
    query3.append(query)

    
dataset.apply(lambda x:query_responsepair3(percent="discount_%",product_name="product_name",row=x,category="sub_category"), axis=1)
query_response_pair_3 = pd.DataFrame({"Query": query3, "Response": response3,"category":category3})

##### 4. I have bugdet of {value}, can i get this product in lower price?

In [162]:
loader = DataFrameLoader(dataset, page_content_column="name")
document=loader.load()
embedding_model = OllamaEmbeddings(model='nomic-embed-text')
db = FAISS.from_documents(document,embedding_model)

In [163]:
def answer_4(df):    
    sim_products = df.apply(lambda x:str(x["product_name"]) +", ratings: " + str(x["ratings"]) + " discount_price: " + str(x["discount_price"]) +
                                " actual_price: " + str(x["actual_price"]) if x["discount_price"]!=None else str(x["product_name"]) +", ratings: " + str(x["ratings"]) + 
                                " actual_price: " + str(x["actual_price"]) + "no discount available", axis=1)
    #print(sim_products)
    comb_product = '\n'.join(f"{j+1}. " + str(x) for j, x in enumerate(sim_products.tolist()))
    return comb_product
def modify1(df,similar_product):
    data = pd.DataFrame()
    for product in similar_product:
        if (product.metadata["discount_price_new"] <value) | (product.metadata["actual_price_new"] <value):
            filter_data = df[(df["name"] == product.page_content) & (df["no_of_ratings"] == product.metadata["no_of_ratings"])]
            data = pd.concat([data,filter_data],ignore_index=True).sort_values("ratings",ascending= False,ignore_index=True)[:5]
    return data
def modify2(df,similar_product):
    data1 = pd.DataFrame()
    for product in similar_product:
        if ((product.metadata["discount_price_new"] <value) | (product.metadata["actual_price_new"] <value)) & (product.metadata["ratings"]>=3.5) & (product.metadata["no_of_ratings"]>50):
            filter_data = df[(df["name"] == product.page_content) & (df["no_of_ratings"] == product.metadata["no_of_ratings"])]
            data1 = pd.concat([data1,filter_data],ignore_index=True)[:5]
    return data1

In [164]:
query4 = []
response4 = []
dis_range = []
category4 = []
def query_responsepair4(product_name, name, actual, percent, discount, row, value, description,category):
    query = f"I have bugdet of {value}, can i get this {row[product_name]} in lower price?"
    act_price = row[actual]
    dis_price = row[discount]
    dis = row[percent]
    category=f"{row[category]}"
    if act_price >= value and dis == 0:
        response = f"""This product comes with a fixed price, but I can offer you free shiping on this product. how does that sound?
                """
        response4.append(response)
        dis_range.append("0")
        category4.append(category)
    elif dis_price > value and dis >= 40:
        similar_product = db.similarity_search(f"similar product of this {row[name]}",k=10)
        data = modify1(dataset,similar_product)
        if data.empty:
            if row[description] != None:
                similar_product = db.similarity_search(f"{' '.join(row[name].split()[1:5]) + ' ' + row[description]}",k=100)
                data1 = modify2(dataset,similar_product)
                
            else:
                similar_product = db.similarity_search(f"{' '.join(row[name].split()[1:5])}",k=100)
                data1 = modify2(dataset,similar_product)
            comb_product = answer_4(data1)
            response = f"""Sorry, i can't reduce price further because it has already been discounted by over 30%. And here are no exact similar product with 
            this bugdet but here are some another product with different brand that match your bugdet\n\n{comb_product}\n\nDo any of them appeal to you?"""
            response4.append(response)
            dis_range.append("greater than or equal to 40") 
            category4.append(category)  
        if not data.empty:
            comb_product = answer_4(data)
            response = f"""Sorry, i can't reduce price further because it has already been discounted by over 30%. Here are some similar products that match your budget.   
\n{comb_product}\n\nDo any of them appeal to you?"""
            response4.append(response)
            dis_range.append("greater than or equal to 40")
            category4.append(category)

    elif dis_price > value and dis > 0 and dis <= 20:
        response = f"""Yes If you are new user then you can get 4% off on discounted price of this product. If you are already an user then you can get an amazing deal with upto 10% off.
Would you like to add this to your cart or explore other options?
                    """
        response4.append(response)
        dis_range.append("0 to 20")
        category4.append(category)

    elif dis_price > value and dis > 20 and dis <= 39:
        response = f"""Yes, If you are new user then you can get 2% off on discounted price of this product. If you are already an user then you can get an amazing deal with upto 5% off.
    Would you like to add this to your cart or explore other options?"""
        response4.append(response)
        dis_range.append("20 to 39")
        category4.append(category)
    else:
        response = None
        response4.append(response)
        dis_range.append("no discount")
        category4.append(category)
    
    query4.append(query)




In [ ]:
for i in category:
    if i == category[0]:
        for value in [30000,40000]:   
            dataset[dataset["sub_category"]==i].apply(lambda x:query_responsepair4(product_name="product_name",description= "description", discount="discount_price_new",actual="actual_price_new",percent="discount_%",name="name",value=value,row=x,category="sub_category"), axis=1)
    elif i == category[1]:
        for value in [50000]:   
            dataset[dataset["sub_category"]==i].apply(lambda x:query_responsepair4(product_name="product_name",description= "description", discount="discount_price_new",actual="actual_price_new",percent="discount_%",name="name",value=value,row=x,category="sub_category"), axis=1)
    elif i == category[2]:
        for value in [100000]:   
            dataset[dataset["sub_category"]==i].apply(lambda x:query_responsepair4(product_name="product_name",description= "description", discount="discount_price_new",actual="actual_price_new",percent="discount_%",name="name",value=value,row=x,category="sub_category"), axis=1)
query_response_pair_4 = pd.DataFrame({"Query": query4, "Response": response4, "discount_range": dis_range,"category":category4})
query_response_pair_4 = query_response_pair_4.dropna()

##### 5. I’ve bought from you before, do I get a special deal?

In [172]:
query5 = []
response5 = []
category5 = []
def query_responsepair5(product_name, percent,row,category):
    dis = row[percent]
    product_name = row[product_name]
    category=f"{row[category]}"
    query = f"I’ve bought from you before, do I get a special deal?"

    if dis==0:
        response = f"""Thank you for your loyalty! As a token of appreciation,\nI am offering you exclusive upto 5% discount on your next purchase. Would you like to apply it now?
"""
        response5.append(response)
        category5.append(category)
    elif dis > 0 and dis <= 30:
        response = f"""Thank you for your loyalty! The product {product_name} is already discounted by {dis}%, As a token of appreciation,\nI am offering you exclusive 5% discount on your next purchase. Would you like to apply it now?
"""
        response5.append(response)
        category5.append(category)
    else:
        response = f"""Thank you for your loyalty! The product {product_name} is already discounted by {dis}% and it's also part of our current sales, As a token of appreciation,\nI am offering you scratch card for an additional reward, would you like to apply it now or your next purchase?
"""
        response5.append(response)
        category5.append(category)

    query5.append(query)
    
dataset.apply(lambda x:query_responsepair5(product_name="product_name",percent="discount_%",row=x,category="sub_category"), axis=1)
query_response_pair_5 = pd.DataFrame({"Query": query5, "Response": response5, "category": category5})

##### 6. Could you show me the products that are part of the sale?\nor Could you show me the products of ongoing festival sale?

In [176]:
query6 = []
response6 = []
category6 = []
def query_responsepair6(df,category):
    query = f"Could you show me the products that are part of the sale?\nor Could you show me the products of ongoing festival sale?"
    category=f"{category}"
    filter_df = df[(df["discount_%"]>40) & (df["ratings"]>=4.0)].sort_values(["no_of_ratings"],ascending= False,ignore_index=True)
    products = filter_df.apply(lambda x:x["product_name"]+" -"+" Now at "+str(x["discount_%"])+"% off",axis=1)[:10]
    comb_product = '\n'.join(f"{j+1}. "+str(x) for j, x in enumerate(products.tolist()))
    response = f"""Sure! here are some of the top product included in our current sales\n\n{comb_product}\n\nwould you like know more detail about product?"""
    response6.append(response)
    query6.append(query)
    category6.append(category)

for i in category:
    query_responsepair6(dataset[dataset["sub_category"]==i],i) 
    

query_response_pair_6 = pd.DataFrame({"Query": query6, "Response": response6,"category":category6})

##### 7. Can you give me free shipping on this order?

In [178]:
query7 = []
response7 = []
def query_responsepair7():
    query = f"Can you give me free shipping on this order?"
    response = f"""You get free shipping on your first purchase. Otherwise you have to order minimum ₹20000 to get free shipping"""
    response7.append(response)
    query7.append(query)
    
query_responsepair7()
query_response_pair_7 = pd.DataFrame({"Query": query7, "Response": response7})

##### 8. Recommend a AC with good rating.

In [179]:
def create_answer(filter_data,v,string1=None):
    data = pd.DataFrame()
    for value in v:
        max = filter_data[filter_data["ratings"]== value]["no_of_ratings"].max()
        filter = filter_data[(filter_data["ratings"]== value) & (filter_data["no_of_ratings"]==max)][:1]
        data = pd.concat([data,filter])
    products = data.apply(lambda x: f"{x['name']} - It has an impressive {str(x['ratings'])} stars rating from {str(x['no_of_ratings'])} ratings",axis=1)
    comb_product = '\n'.join("> "+str(x) for x in products.tolist())
    response = f"""{string1}\n\n{comb_product}\n\nwould you like know more detail about product?"""
    return response
    

In [ ]:
query8 = []
response8 = []
def query_responsepair8():
    Question = ["Can you recommend a AC or Air Conditioners or Air Conditioner with good ratings?","Could you show me the top-rated AC or Air Conditioners or Air Conditioner that available?",
                "Can you recommend a Air Cooler with good ratings?","Could you show me the top-rated Air Cooler that available?",
                "Can you recommend a Refrigerator or Refrigerators with good ratings?","Could you show me the top-rated Refrigerator or Refrigerators that available?",
                "Can you recommend a TV or tv or LED Smart TV with good ratings?","Could you show me the top-rated TV or tv or LED Smart TV that available?"]
    for query in Question:
        query8.append(query)
        if any(word  in query.lower() for word in ["air conditioner","ac","air conditioners"]):
            filter_data=dataset[dataset["name"].apply(lambda x: any(substr in str(x) for substr in ["AC","Air Conditioner","Air Conditioners"]))]
            v1 = filter_data["ratings"].sort_values(ascending = False).unique().tolist()[:10]
            if "recommend" in query.lower():
                res = create_answer(filter_data,v1,string1="Absolutely! I can Recommend some product with good rating. here are the product...")
                response8.append(res)
            else:
                res = create_answer(filter_data,v1,string1="Certainly! Here are some of the best-rated product we have right now...")
                response8.append(res)
        elif any(word  in query.lower() for word in["air cooler"]):
            filter_data=dataset[dataset["name"].apply(lambda x: any(substr in str(x) for substr in ["Air Cooler"]))]
            v2 = filter_data["ratings"].sort_values(ascending = False).unique().tolist()[:10]
            if "recommend" in query.lower():
                res = create_answer(filter_data,v2,string1="Absolutely! I can Recommend some product with good rating. here are the product...")
                response8.append(res)
            else:
                res = create_answer(filter_data,v2,string1="Certainly! Here are some of the best-rated product we have right now...")
                response8.append(res)
        elif any(word  in query.lower() for word in["refrigerator"]):
            filter_data=dataset[dataset["name"].apply(lambda x: any(substr in str(x) for substr in ["Refrigerator","Refrigerators"]))]
            v3 = filter_data["ratings"].sort_values(ascending = False).unique().tolist()[:10]
            if "recommend" in query.lower():
                res = create_answer(filter_data,v3,string1="Absolutely! I can Recommend some product with good rating. here are the product...")
                response8.append(res)
            else:
                res = create_answer(filter_data,v2,string1="Certainly! Here are some of the best-rated product we have right now...")
                response8.append(res)
        elif any(word  in query.lower() for word in["tv","led Smart tv"]):
            filter_data=dataset[dataset["name"].apply(lambda x: any(substr in str(x) for substr in ["TV","tv","LED Smart TV"]))]
            v4 = filter_data["ratings"].sort_values(ascending = False).unique().tolist()[:10]
            if "recommend" in query.lower():
                res = create_answer(filter_data,v4,string1="Absolutely! I can Recommend some product with good rating. here are the product...")
                response8.append(res)
            else:
                res = create_answer(filter_data,v2,string1="Certainly! Here are some of the best-rated product we have right now...")
                response8.append(res)
    
query_responsepair8()
query_response_pair_8 = pd.DataFrame({"Query": query8, "Response": response8})

##### 9. Recommend AC, Air cooler or Air Conditioner

In [350]:
from sklearn.preprocessing import RobustScaler
new_data = pd.DataFrame([
            {"name": "Air Cooler","ratings":0,"no_of_ratings":0,"product_name":"Air Cooler"},
            {"name": "Portable Air Conditioner","ratings":0,"no_of_ratings":0,"product_name":"Portable Air Conditioner"},
            {"name": "5 star Ac with good feature","ratings":0,"no_of_ratings":0,"product_name":"5 star Ac with good feature"},
            {"name": "Show me split ACs with inverter technology.","ratings":0,"no_of_ratings":0,"product_name":"Inverter Split AC"},
            {"name":"which AC has the best cooling mode?","ratings":0,"no_of_ratings":0,"product_name":"which AC has the best cooling mode?"},
            {"name": "I need a small refrigerator for a small kitchen.","ratings":0,"no_of_ratings":0,"product_name":"need a small refrigerator for a small kitchen."},
            {"name": "I’m looking for a Whirlpool or Samsung refrigerator","ratings":0,"no_of_ratings":0,"product_name":"I’m looking for a Whirlpool or Samsung Refrigerator"},
            {"name": "Show me refrigerators with a double door.","ratings":0,"no_of_ratings":0,"product_name":"Show me refrigerators with a double door."},
            {"name":"Show me 4K TV with a screen size of at least 100 inches.","ratings":0,"no_of_ratings":0,"product_name":"Show me 4K TV with a screen size of at least 100 inches"},
            {"name": "I’m looking for a Daikin or Carrier air conditioner.","ratings":0,"no_of_ratings":0,"product_name":"I’m looking for a Daikin or Carrier AC."},
            {"name": "I’m looking for a Samsung and LG TV.","ratings":0,"no_of_ratings":0,"product_name":"I’m looking for a Samsung and LG TV."},
            ])

new_df = pd.concat([dataset,new_data], ignore_index= True)

scaler = RobustScaler()
new_df.loc[new_df.index,['new_ratings', 'new_no_of_ratings']] = scaler.fit_transform(
    new_df[['ratings', 'no_of_ratings']])

In [351]:
def weighted_score(df):
    # Define weights for each feature
    rating_weight = 0.3
    no_of_ratings_weight= 0.7

    # Compute a score based on the weighted combination of rating and price
    
    df.loc[df.index,'final_score'] = np.where(df["ratings"]<=4.4,
        (rating_weight * df['new_ratings'] +
        no_of_ratings_weight * df['new_no_of_ratings']), np.sqrt(np.square( df['ratings'])+np.square(df['no_of_ratings']))
    )
    return df
def answer(similar_product,df):
    final_df = pd.DataFrame()
    data1 = pd.DataFrame()
    data2 = pd.DataFrame()
    for product in similar_product:
        if (product.metadata["ratings"]<=4.4):
            filter_data1 = df[(df["name"] ==product.page_content) & (df["no_of_ratings"] == product.metadata["no_of_ratings"])]
            data1 = pd.concat([data1,filter_data1],ignore_index=True)
        if (product.metadata["ratings"]>4.4):
            filter_data2 = df[(df["name"] ==product.page_content) & (df["no_of_ratings"] == product.metadata["no_of_ratings"])]
            data2 = pd.concat([data2,filter_data2],ignore_index=True)
    
    modified_data1 = weighted_score(data1).sort_values(by='final_score', ascending=False)[:7].reset_index(drop=True)
    final_df = pd.concat([final_df,modified_data1],ignore_index=True)
    if not data2.empty:
        modified_data2 = weighted_score(data2).sort_values(by='final_score', ascending=False)[:3].reset_index(drop=True)
        final_df = pd.concat([final_df,modified_data2],ignore_index=True)
    
    final_recommendation = final_df.apply(lambda x:x['product_name']+" - " + x["description"]+ " (rating: "+str(x['ratings'])+", no. of ratings: "+str(x['no_of_ratings'])+")"
                        if x["description"] != None else x['product_name']+" (rating: "+str(x['ratings'])+", no. of ratings: "+str(x['no_of_ratings'])+")", axis=1)
    comb_product = '\n'.join(f"{j+1}. " + str(x) for j, x in enumerate(final_recommendation.tolist()))
    return comb_product

In [352]:
query9 = []
response9 = []
category9 =[]
def top_recommendation(data,product_name,category):
    for i in data.index:
        product_index = i
        if product_index < 449:
            query = f"Recommend similar product of this '{data[product_name][product_index]}' product"
            category=f"{category}"
            query9.append(query)
            category9.append(category)
            similar_product = db.similarity_search(f"{data[product_name][product_index]}",k=30)
            comb_product = answer(similar_product,new_df)
            response = f""" Here are some product similar to {data[product_name][product_index]}\n\n{comb_product}\n\nyou might like these products, Would you be interested in checking them out?"""
            response9.append(response)
        elif product_index >= 449 and product_index  <= 471:
            query = f"Recommend similar product of this {data[product_name][product_index]} product"
            category=f"{category}"
            query9.append(query)
            category9.append(category)
            similar_product = db.similarity_search(f"{data[product_name][product_index]}",k=20)
            comb_product = answer(similar_product,new_df)
            response = f"""Here are some product similar to  {data[product_name][product_index]}...\n\n{comb_product}\n\nyou might like them! Would you be interested in checking them out?"""
            response9.append(response)
        elif product_index >471 and product_index <= 1341:
            query = f"Recommend similar product of this '{data[product_name][product_index]}' product"
            category=f"{category}"
            query9.append(query)
            category9.append(category)
            similar_product = db.similarity_search(f"{data[product_name][product_index]}",k=30)
            comb_product = answer(similar_product,new_df)
            response = f""" Here are some product similar to {data[product_name][product_index]}\n\n{comb_product}\n\nyou might like these products, Would you be interested in checking them out?"""
            response9.append(response)
        elif product_index > 1341 and product_index  <= 1344:
            query = f"Can you Recommend {data[product_name][product_index]}"
            category=f"{category}"
            query9.append(query)
            category9.append(category)
            similar_product = db.similarity_search(f"{data[product_name][product_index]}",k=100)
            comb_product = answer(similar_product,new_df)
            response = f"""Yes! I recommed some {data[product_name][product_index]}...\n\n{comb_product}\n\nyou might like them! Would you be interested in checking them out?"""
            response9.append(response)
        elif product_index > 1344 and product_index<1351:
            query = f"{data[product_name][product_index]}"
            category=f"{category}"
            query9.append(query)
            category9.append(category)
            similar_product = db.similarity_search(f"{data[product_name][product_index]}",k=100)
            comb_product = answer(similar_product,new_df)
            response = f"""Sure! Here are some great options you are looking for {data[product_name][product_index]}...\n\n{comb_product}\n\nnWould you like more details about any specific model? Let me know"""
            response9.append(response)
        elif product_index >=1351:
            query = f"{data[product_name][product_index]}"
            category=f"{category}"
            query9.append(query)
            category9.append(category)
            similar_product = db.similarity_search(f"{data[product_name][product_index]}",k=60)
            comb_product = answer(similar_product,new_df)
            response = f"""Sure! Here are some great options you are looking for {data[product_name][product_index]}...\n\n{comb_product}\n\nWould you like more details about any specific model? Let me know!"""
            response9.append(response)

for i in category:
    top_recommendation(new_df[new_df["sub_category"]==i],"product_name",i)
top_recommendation(new_df[1342:],"product_name","sub_category")
query_response_pair_9 = pd.DataFrame({"Query": query9, "Response": response9, "category":category9})

##### 10. Apply discount or coupon code 

In [360]:
query10 = []
response10 = []
def query_responsepair10():
    q1 = "Apply discount now"
    q2 = "Apply coupon code Now"
    if q1:
        query10.append(q1)
        response = f"Ok, Discount has been applied for this purchase."
        response10.append(response)
    if q2:
        query10.append(q2)
        response = f"Ok, Coupon code has been applied for this purchase."
        response10.append(response)

query_responsepair10()
query_response_pair_10 = pd.DataFrame({"Query": query10, "Response": response10})

In [442]:
def modified_product(similar_product,rating = None,value = None):
    data = pd.DataFrame()
    for product in similar_product:
        filter_data = dataset[(dataset["name"] ==product.page_content) & (dataset["no_of_ratings"] == product.metadata["no_of_ratings"])]
        data = pd.concat([data,filter_data],ignore_index=True)
    if value !=None:
        filter_data = data[(data["ratings"]>rating)&(data["actual_price_new"]<value)|(data["discount_price_new"]<value)].sort_values("final_score",ascending=False)[:15].reset_index(drop=True)
    elif value == None:
        filter_data = data[(data["ratings"]>rating)].sort_values("final_score",ascending=False)[:15].reset_index(drop=True)    
    products = filter_data.apply(lambda x:f"""{x['product_name']}\n -rating: {str(x['ratings'])} ({str(x['no_of_ratings'])} reviews)\n-original price: {x['actual_price']}, Now: {x['discount_price']} ({x['discount_%']}% off)\n""" if x['discount_price_new'] != None 
else f"""{x['product_name']}\n -rating: {str(x['ratings'])} (no. of ratings: {str(x['no_of_ratings'])})\n -price: {x['actual_price']}\n""", axis=1)
    comb_product = '\n'.join(f"{j+1}. " + str(x) for j, x in enumerate(products.tolist()))
    return comb_product

In [443]:
query11 = []
response11 = []
def query_responsepair11(df,v1,v2,v3,v4):
    q0= f"can you suggest some AC with 1.5 or 2 Ton capacity?"
    q1 = f"What are the best AC under ₹{v1}?" 
    q2 = f"what are the best Air Cooler under ₹{v2}?"
    q3 = f"What are the best Tv under ₹{v3}?" 
    q4 = f"what are the best Refrigerator under ₹{v4}?"
    q5= f"can you suggest any 8K Ultra HD Smart NEO or neo QLED TV? "
    df.loc[df.index,'final_score'] = (
        df["ratings"] * np.log(0.5 + df["no_of_ratings"])
    )
    if q0 :
        query11.append(q0)
        similar_product = db.similarity_search(q0,k=100)
        comb_product = modified_product(similar_product=similar_product,rating=4.0)
        response = f"""Sure! Here are some amazing AC with 1.5 or 2 ton capacity \n\n{comb_product}\n\n Would you like any of them for Add in your cart? or Let me know if you'd like more information or if you'd like to see more options!"""
        response11.append(response)
    if q5:
        query11.append(q5)
        similar_product = db.similarity_search(q5,k=30)
        comb_product = modified_product(similar_product=similar_product,rating=4.0)
        response = f"""Sure! Here are Some amazing smart TV\n\n{comb_product}\n\n Would you like any of them for Add in your cart? or Let me know if you'd like more information or if you'd like to see more options!"""
        response11.append(response)
    for i in [q1,q3,q4]:
        query11.append(i)
        similar_product = db.similarity_search(i,k=150)
        if i == q1:
            comb_product = modified_product(similar_product=similar_product,rating=4.0,value=v1)
            response = f"""Sure! Based on your preference, Here are Some amazing products under ₹{v1}\n\n{comb_product}\n\n Would you like any of them for Add in your cart? or Let me know if you'd like more information or if you'd like to see more options!"""
            response11.append(response)
        if i == q3:
            comb_product = modified_product(similar_product=similar_product,rating=4.0,value=v3)
            response = f"""Sure! Based on your preference, Here are Some amazing products under ₹{v1}\n\n{comb_product}\n\n Would you like any of them for Add in your cart? or Let me know if you'd like more information or if you'd like to see more options!"""
            response11.append(response)
        if i == q4:
            comb_product = modified_product(similar_product=similar_product,rating=4.0,value=v4)
            response = f"""Sure! Based on your preference, Here are Some amazing products under ₹{v1}\n\n{comb_product}\n\n Would you like any of them for Add in your cart? or Let me know if you'd like more information or if you'd like to see more options!"""
            response11.append(response)

    if q2:
        query11.append(q2)
        similar_product = db.similarity_search(q2,k=30)
        comb_product = modified_product(similar_product=similar_product,value=v2)
        response = f"""Sure! Based on your preference, Here are Some amazing products under ₹{v2}\n\n{comb_product}\n\n Would you like any of them for Add in your cart? or Let me know if you'd like more information or if you'd like to see more options!"""
        response11.append(response)
        

query_responsepair11(dataset,v1=50000,v2=10000,v3=50000,v4=40000)
query_response_pair_11 = pd.DataFrame({"Query": query11, "Response": response11})

In [462]:
Q_R9 = pd.concat([query_response_pair_9[:1342].groupby("category").sample(frac=0.15,random_state=55),query_response_pair_9[1342:]])

In [461]:
Q_R5 = query_response_pair_5.groupby("category").sample(frac=0.15,random_state=11)

In [459]:
Q_R4=query_response_pair_4.groupby(["category","discount_range"]).sample(n=30,replace= True,random_state=99)

In [455]:
Q_R3 = query_response_pair_3.groupby("category").sample(frac=0.15,random_state=66)

In [453]:
Q_R2=query_response_pair_2.groupby(["category","discount_range"]).sample(n=30,replace= True,random_state=33)

In [451]:
Q_R1 = query_response_pair_1.groupby("category").sample(frac=0.15,random_state=42)

In [465]:
Q_R6 = query_response_pair_6
Q_R7 = query_response_pair_7
Q_R8=  query_response_pair_8
Q_R10 = query_response_pair_10
Q_R11 = query_response_pair_11

In [ ]:
Q_R = pd.concat([Q_R1,Q_R2,Q_R3,Q_R4,Q_R5,Q_R6,Q_R7,Q_R8,Q_R9,Q_R10,Q_R11],ignore_index=True)
Q_R_final = Q_R.sample(frac=1,random_state=42).reset_index(drop=True)
Q_R_final.drop(["discount_range","category"], axis=1, inplace=True)
Q_R_final.to_csv("Q_R.csv",index=False)